In [1]:
# ============================================================
#  Kcbert 광고 분류 파인튜닝
#  기반 모델 : beomi/Kcbert-base
#  분류 목표 : review_body → is_ad (0: 비광고, 1: 광고)
#  탐색 방식 : Grid Search (54 조합)
#  평가 기준 : Recall 1순위, F1-score 2순위
# ============================================================

# ── 0. 패키지 설치 (Colab 최초 1회) ──────────────────────────
!pip install transformers datasets scikit-learn pandas torch -q

In [2]:
# ── 1. 임포트 ─────────────────────────────────────────────────
import re
import os
import random
import itertools
import warnings
from html import unescape

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score, f1_score, precision_score, accuracy_score,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

In [3]:
# ── 2. 시드 고정 ───────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()

In [4]:
# ── 3. 전처리 함수 ─────────────────────────────────────────────
def preprocess(text: str) -> str:
    """
    review_body 전처리
    1) HTML 엔티티 디코딩  (&amp; → &)
    2) HTML 태그 제거      (<b>텍스트</b> → 텍스트)
    3) 해시태그 단어 추출  (#맛집 → 맛집)  ← 광고 피처 보존
    4) 말줄임 제거         (... → 공백)
    5) 특수문자 정리       (한글/영문/숫자/기본문장부호만 유지)
    6) 과도한 공백 정리
    """
    if not isinstance(text, str):
        return ""
    text = unescape(text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"#(\w+)", r"\1 ", text)
    text = re.sub(r"\.{2,}", " ", text)
    text = re.sub(r"[^\w\s가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9.,!?~]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
# ── 4. CSV 로드 및 전처리 ──────────────────────────────────────
CSV_PATH = "/content/CrawlingReviewList_rows.csv"   # ← 실제 파일 경로로 변경하세요

print("=" * 60)
print("[1] 데이터 로드 및 전처리")
print("=" * 60)

df = pd.read_csv(CSV_PATH)
print(f"  원본 행 수       : {len(df)}")

# 필요 컬럼만 추출
df = df[["review_body", "is_ad"]].copy()

# 결측값 제거
df.dropna(subset=["review_body", "is_ad"], inplace=True)

# 레이블 정수형 변환
df["is_ad"] = df["is_ad"].astype(int)

# 전처리 적용
df["review_body"] = df["review_body"].apply(preprocess)

# 빈 텍스트 제거 (전처리 후 빈 문자열)
df = df[df["review_body"].str.len() > 0].reset_index(drop=True)

print(f"  전처리 후 행 수  : {len(df)}")
print(f"\n  레이블 분포")
label_counts = df["is_ad"].value_counts().sort_index()
for label, count in label_counts.items():
    label_name = "광고" if label == 1 else "비광고"
    print(f"    {label} ({label_name}) : {count}건  ({count/len(df)*100:.1f}%)")

[1] 데이터 로드 및 전처리
  원본 행 수       : 1051
  전처리 후 행 수  : 1050

  레이블 분포
    0 (비광고) : 729건  (69.4%)
    1 (광고) : 321건  (30.6%)


In [6]:
# ── 5. Train / Test 분할 ──────────────────────────────────────
print("\n" + "=" * 60)
print("[2] Train / Test 분할 (8:2, Stratified)")
print("=" * 60)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["is_ad"],  # 클래스 비율 유지
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"  Train : {len(train_df)}건")
print(f"  Test  : {len(test_df)}건")


[2] Train / Test 분할 (8:2, Stratified)
  Train : 840건
  Test  : 210건


In [7]:
# ── 6. 클래스 가중치 계산 (불균형 대응) ───────────────────────
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["is_ad"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"\n  클래스 가중치 → 비광고(0): {class_weights[0]:.4f} / 광고(1): {class_weights[1]:.4f}")


  클래스 가중치 → 비광고(0): 0.7204 / 광고(1): 1.6342


In [8]:
# ── 7. Dataset 클래스 ──────────────────────────────────────────
MODEL_NAME = "beomi/Kcbert-base"
MAX_LEN    = 256

class AdDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "token_type_ids": self.encodings.get(
                "token_type_ids",
                torch.zeros_like(self.encodings["input_ids"])
            )[idx],
            "labels": self.labels[idx],
        }

In [9]:
# ── 8. 평가 함수 ───────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "recall":    recall_score(labels, preds, pos_label=1, zero_division=0),
        "f1":        f1_score(labels, preds, pos_label=1, zero_division=0),
        "precision": precision_score(labels, preds, pos_label=1, zero_division=0),
        "accuracy":  accuracy_score(labels, preds),
    }

In [10]:
# ── 9. 클래스 가중치 적용 커스텀 Trainer ──────────────────────
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [11]:
# ── 10. 하이퍼파라미터 그리드 ─────────────────────────────────
LEARNING_RATES = [1e-5, 3e-5, 5e-5]
SCHEDULERS     = ["linear", "cosine", "cosine_with_restarts"]
DROPOUTS       = [0.1, 0.2, 0.3]
BATCH_SIZES    = [16, 32]
EPOCHS         = 10

grid = list(itertools.product(LEARNING_RATES, SCHEDULERS, DROPOUTS, BATCH_SIZES))

print("\n" + "=" * 60)
print("[3] Grid Search 시작")
print(f"    총 실험 조합 : {len(grid)}가지")
print(f"    최대 Epoch   : {EPOCHS} (Early Stopping 적용)")
print("=" * 60)


[3] Grid Search 시작
    총 실험 조합 : 54가지
    최대 Epoch   : 10 (Early Stopping 적용)


In [12]:
# ── 11. 토크나이저 로드 (1회만) ────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test Dataset은 고정 (매 실험 동일)
test_dataset = AdDataset(test_df["review_body"], test_df["is_ad"], tokenizer)

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
# ── 12. Grid Search 루프 ───────────────────────────────────────
results = []
RESULTS_PATH = "bert2crawling_results_all.csv"

for exp_idx, (lr, scheduler, dropout, batch_size) in enumerate(grid, start=1):

    print(f"\n[실험 {exp_idx:02d}/{len(grid)}]  "
          f"lr={lr}  scheduler={scheduler}  "
          f"dropout={dropout}  batch={batch_size}")

    set_seed()  # 매 실험마다 시드 재고정

    # Train Dataset 구성
    train_dataset = AdDataset(
        train_df["review_body"], train_df["is_ad"], tokenizer
    )

    # 모델 초기화 (dropout 적용)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
        ignore_mismatched_sizes=True,
    )

    # TrainingArguments
    training_args = TrainingArguments(
        output_dir=f"./ckpt/exp_{exp_idx:02d}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        lr_scheduler_type=scheduler,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="recall",   # Recall 기준으로 best 선택
        greater_is_better=True,
        logging_steps=50,
        seed=SEED,
        fp16=torch.cuda.is_available(),   # GPU 있으면 FP16 사용
        report_to="none",                 # wandb 등 비활성화
    )

    # WeightedTrainer
    trainer = WeightedTrainer(
        class_weights=class_weights_tensor,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    # 학습
    trainer.train()

       # ── log_history에서 epoch별 지표 추출 ─────────────────────
    log_history    = trainer.state.log_history
    train_logs     = [x for x in log_history if "loss" in x and "eval_loss" not in x]
    eval_loss_logs = [x for x in log_history if "eval_loss" in x]
    eval_logs      = [x for x in log_history if "eval_recall" in x]

    # 전체 best 결과 (recall 최고 epoch 기준)
    best_eval = max(eval_logs, key=lambda x: x["eval_recall"])
    recall    = best_eval.get("eval_recall",    0)
    f1        = best_eval.get("eval_f1",        0)
    precision = best_eval.get("eval_precision", 0)
    accuracy  = best_eval.get("eval_accuracy",  0)

    print(f"  → Recall={recall:.4f}  F1={f1:.4f}  "
          f"Precision={precision:.4f}  Accuracy={accuracy:.4f}")

    # ── row 구성 (전체 best + epoch별 상세) ───────────────────
    row = {
        "exp_id":        exp_idx,
        "learning_rate": lr,
        "scheduler":     scheduler,
        "dropout":       dropout,
        "batch_size":    batch_size,
        "recall":        round(recall,    4),
        "f1":            round(f1,        4),
        "precision":     round(precision, 4),
        "accuracy":      round(accuracy,  4),
    }

    # epoch별 상세 지표 추가
    for i in range(EPOCHS):
        ep = i + 1

        row[f"epoch{ep}_train_loss"] = (
            round(train_logs[i].get("loss", 0), 4)
            if i < len(train_logs) else None
        )
        row[f"epoch{ep}_val_loss"] = (
            round(eval_loss_logs[i].get("eval_loss", 0), 4)
            if i < len(eval_loss_logs) else None
        )
        row[f"epoch{ep}_batch_size"] = (
            batch_size if i < len(eval_loss_logs) else None
        )
        row[f"epoch{ep}_recall"] = (
            round(eval_logs[i].get("eval_recall", 0), 4)
            if i < len(eval_logs) else None
        )
        row[f"epoch{ep}_f1"] = (
            round(eval_logs[i].get("eval_f1", 0), 4)
            if i < len(eval_logs) else None
        )
        row[f"epoch{ep}_precision"] = (
            round(eval_logs[i].get("eval_precision", 0), 4)
            if i < len(eval_logs) else None
        )
        row[f"epoch{ep}_accuracy"] = (
            round(eval_logs[i].get("eval_accuracy", 0), 4)
            if i < len(eval_logs) else None
        )

    # 결과 저장
    results.append(row)

    # 실험마다 즉시 CSV 저장 (런타임 끊겨도 복구 가능)
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")

    # 메모리 정리
    del model, trainer, train_dataset
    torch.cuda.empty_cache()


[실험 01/54]  lr=1e-05  scheduler=linear  dropout=0.1  batch=16


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: beomi/Kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b

Epoch,Training Loss,Validation Loss,Recall,F1,Precision,Accuracy
1,0.699496,0.592181,0.843750,0.593407,0.457627,0.647619
2,0.541603,0.488506,0.718750,0.676471,0.638889,0.790476
3,0.353743,0.436359,0.750000,0.727273,0.705882,0.828571


...

Epoch,Training Loss,Validation Loss,Recall,F1,Precision,Accuracy
1,0.663332,0.599599,0.843750,0.571429,0.432000,0.614286
2,0.474392,0.549809,0.468750,0.606061,0.857143,0.814286
3,0.291477,0.475841,0.781250,0.714286,0.657895,0.809524


  → Recall=0.8438  F1=0.5714  Precision=0.4320  Accuracy=0.6143

[실험 38/54]  lr=5e-05  scheduler=linear  dropout=0.1  batch=32


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: beomi/Kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b

Epoch,Training Loss,Validation Loss,Recall,F1,Precision,Accuracy
1,0.691371,0.582835,0.796875,0.579545,0.455357,0.647619


In [ ]:
# ── 13. 결과 출력 ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("[4] 전체 실험 결과 요약")
print("=" * 60)

results_df = pd.DataFrame(results).sort_values(
    ["recall", "f1"], ascending=False
).reset_index(drop=True)

# 핵심 컬럼만 출력
summary_cols = ["exp_id", "learning_rate", "scheduler", "dropout",
                "batch_size", "recall", "f1", "precision", "accuracy"]
print(results_df[summary_cols].to_string(index=False))

print("\n" + "=" * 60)
print("[5] Recall 기준 Top 5 조합")
print("=" * 60)
print(results_df[summary_cols].head(5).to_string(index=False))

In [ ]:
# ── 14. 최적 모델 재학습 및 저장 ──────────────────────────────
print("\n" + "=" * 60)
print("[6] 최적 조합으로 최종 모델 저장")
print("=" * 60)

best = results_df.iloc[0]
print(f"\n  최적 조합")
print(f"    Learning Rate : {best['learning_rate']}")
print(f"    Scheduler     : {best['scheduler']}")
print(f"    Dropout       : {best['dropout']}")
print(f"    Batch Size    : {int(best['batch_size'])}")
print(f"    Recall        : {best['recall']}")
print(f"    F1-score      : {best['f1']}")

set_seed()

# 전체 데이터로 최종 재학습
full_dataset = AdDataset(df["review_body"], df["is_ad"], tokenizer)

best_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    hidden_dropout_prob=float(best["dropout"]),
    attention_probs_dropout_prob=float(best["dropout"]),
    ignore_mismatched_sizes=True,
)

best_args = TrainingArguments(
    output_dir="./bert2crawling_best_model",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=int(best["batch_size"]),
    learning_rate=float(best["learning_rate"]),
    lr_scheduler_type=best["scheduler"],
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

best_trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=best_model,
    args=best_args,
    train_dataset=full_dataset,
    compute_metrics=compute_metrics,
)
best_trainer.train()

# 최종 Test 평가 출력
final_eval = best_trainer.evaluate(test_dataset)
print("\n  [최종 모델 Test 평가]")
print(f"    Recall    : {final_eval.get('eval_recall',    0):.4f}")
print(f"    F1-score  : {final_eval.get('eval_f1',        0):.4f}")
print(f"    Precision : {final_eval.get('eval_precision', 0):.4f}")
print(f"    Accuracy  : {final_eval.get('eval_accuracy',  0):.4f}")

# 상세 분류 리포트
preds_output = best_trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=-1)
print("\n  [Classification Report]")
print(classification_report(
    test_df["is_ad"].values, preds,
    target_names=["비광고(0)", "광고(1)"]
))

# 모델 & 토크나이저 저장
best_model.save_pretrained("bert2crawling_best_model")
tokenizer.save_pretrained("bert2crawling_tokenizer")

print("\n  저장 완료")
print("    bert2crawling_best_model/")
print("    bert2crawling_tokenizer/")
print("    bert2crawling_results_all.csv")
print("\n" + "=" * 60)
print("  파인튜닝 완료!")
print("=" * 60)

In [ ]:
# ── 15. 저장된 모델 사용 예시 ──────────────────────────────────
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
#
# tokenizer = AutoTokenizer.from_pretrained("bert2crawling_tokenizer")
# model = AutoModelForSequenceClassification.from_pretrained("bert2crawling_best_model")
# model.eval()
#
# text = "정말 맛있었어요! #광고 #협찬"
# inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
# with torch.no_grad():
#     logits = model(**inputs).logits
# pred = torch.argmax(logits, dim=-1).item()
# print("광고" if pred == 1 else "비광고")